In [1]:
import json
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd

import config
from src.db_io import leer_tabla_sqlite
from src.features_modelo import features_modelo_a
from src.feature_tests import (
    benjamini_hochberg, calcular_vif, calcular_woe_iv, chi2_y_cramer,
    clasificar_iv, mann_whitney,
)
from src.log_decisiones import registrar_decision

(config.OUTPUTS_DIR / "eda").mkdir(parents=True, exist_ok=True)

df = leer_tabla_sqlite(config.ORO_DB, "cliente_features")
y = df["etiqueta_adopcion"]

# Candidatas = predictoras del Modelo A + las tres demograficas de SS6.4.
# desc_genero NO es candidata a entrar al modelo, pero SI se mide (SS4 y SS6.4).
candidatas = sorted(set(features_modelo_a(df.columns)) | {"desc_genero"})

continuas = [c for c in candidatas if df[c].dtype.kind in "biufc"]
categoricas = [c for c in candidatas if c not in continuas]

print(f"{len(candidatas)} variables a validar: {len(continuas)} continuas, "
      f"{len(categoricas)} categoricas")
print(f"Demograficas incluidas en la medicion: "
      f"{[c for c in ['desc_genero','grupo_edad','desc_tipo_de_vivienda'] if c in candidatas]}")

64 variables a validar: 60 continuas, 4 categoricas
Demograficas incluidas en la medicion: ['desc_genero', 'grupo_edad', 'desc_tipo_de_vivienda']


In [2]:
filas = []
woe_frames = []

for col in candidatas:
    es_continua = col in continuas
    iv, tabla_woe = calcular_woe_iv(df[col], y)
    tabla_woe.insert(0, "variable", col)
    woe_frames.append(tabla_woe)

    fila = {
        "variable": col,
        "tipo": "continua" if es_continua else "categorica",
        "n_nulos": int(df[col].isnull().sum()),
        "iv": iv,
        "clase_iv": clasificar_iv(iv),
        "p_mann_whitney": np.nan,
        "chi2": np.nan,
        "p_chi2": np.nan,
        "v_cramer": np.nan,
    }
    if es_continua:
        mw = mann_whitney(df[col], y)
        fila["p_mann_whitney"] = mw["p_valor"]
        fila["mediana_adoptantes"] = mw["mediana_evento"]
        fila["mediana_no_adoptantes"] = mw["mediana_no_evento"]
    else:
        ch = chi2_y_cramer(df[col], y)
        fila.update({"chi2": ch["chi2"], "p_chi2": ch["p_valor"],
                     "v_cramer": ch["v_cramer"]})
    filas.append(fila)

validacion = pd.DataFrame(filas)
woe_por_bin = pd.concat(woe_frames, ignore_index=True)
print(validacion.sort_values("iv", ascending=False).head(20).to_string(index=False))

                         variable     tipo  n_nulos       iv clase_iv  p_mann_whitney  chi2  p_chi2  v_cramer  mediana_adoptantes  mediana_no_adoptantes
          n_productos_no_etiqueta continua        0 2.149881   fuerte             0.0   NaN     NaN       NaN        2.000000e+00           1.000000e+00
              saldo_liquido_total continua        0 1.838402   fuerte             0.0   NaN     NaN       NaN        3.850014e+06           0.000000e+00
        antiguedad_relacion_meses continua   331037 1.615513   fuerte             0.0   NaN     NaN       NaN        1.200000e+01           1.100000e+01
           dias_desde_ultimo_dato continua   332063 1.586888   fuerte             0.0   NaN     NaN       NaN        0.000000e+00           3.100000e+01
     cuenta_ahorro_saldo_snapshot continua        0 1.484693   fuerte             0.0   NaN     NaN       NaN        3.169799e+06           0.000000e+00
        ratio_liquidez_patrimonio continua   164591 1.442432   fuerte             

In [3]:
# Un p-valor por variable: el de Mann-Whitney si es continua, el de chi2 si es
# categorica. Se corrigen TODOS juntos, porque el problema de multiplicidad es
# sobre el conjunto de variables probadas, no por familia de test.
validacion["p_valor"] = validacion["p_mann_whitney"].fillna(validacion["p_chi2"])

con_p = validacion["p_valor"].notna()
q, rechaza = benjamini_hochberg(validacion.loc[con_p, "p_valor"].to_numpy(), alpha=0.05)
validacion.loc[con_p, "q_bh"] = q
validacion.loc[con_p, "significativa_fdr"] = rechaza

n_crudo = int((validacion["p_valor"] < 0.05).sum())
n_fdr = int(validacion["significativa_fdr"].fillna(False).sum())
print(f"Significativas sin corregir: {n_crudo} / {int(con_p.sum())}")
print(f"Significativas tras Benjamini-Hochberg (FDR 5%): {n_fdr}")
# Con n=860,223 casi todos los p-valores ya nacen ~0 (hasta diferencias minimas
# son "significativas" a este tamano de muestra), asi que BH tiene poco margen
# para tumbar nada: el conteo antes y despues coincide o casi coincide. La
# clasificacion por IV (celda 1), que sí pesa el tamano del efecto, es la que
# hace el trabajo real de discriminar variables utiles de las que no lo son.


Significativas sin corregir: 61 / 64
Significativas tras Benjamini-Hochberg (FDR 5%): 61


In [4]:
# Atencion especial a patrimonio / activos / pasivos, relacionados por
# definicion contable (patrimonio = activos - pasivos).
#
# `calcular_vif` hace un dropna() CONJUNTO sobre todo el batch (una sola fila
# completa por cliente). Si metemos en ese mismo batch una bandera 0/1 que
# documenta la ausencia de OTRA columna del batch (p.ej. `falta_estimador` =
# `estimador_ingreso`.isnull()), el dropna conjunto elimina exactamente las
# filas donde esa bandera vale 1 -- en la submuestra que sobrevive la bandera
# queda CONSTANTE en 0. Una columna constante regresada con intercepto da
# R^2~=1, o sea VIF=inf: eso mide como se construyo la muestra, no
# colinealidad real de la poblacion. Por eso las banderas de
# faltante/disponibilidad de dato se excluyen de esta matriz (no de la tabla
# de validacion completa, solo del calculo de VIF); las banderas de tenencia
# de producto (`*_tenencia`) SI se dejan, porque no son bookkeeping de nulos
# de otra columna del batch -- son señal de negocio (saldo_snapshot del mismo
# producto se rellena con 0.0 incondicionalmente, nunca queda NaN, así que no
# hay mecanismo de dropna conjunto que las fuerce a una constante).
BANDERAS_FALTANTE_VIF = {
    "falta_estimador", "tiene_estimador_ingreso",
    "sin_dato_financiero", "sin_dato_financiero_total", "falta_financiero",
    "sin_dato_reciente",
    "falta_vivienda", "tiene_dato_vivienda",
    "cv_saldo_liquido_insuficiente",
}
cols_vif = [c for c in continuas
            if df[c].notna().mean() > 0.5 and c not in BANDERAS_FALTANTE_VIF]
muestra = df[cols_vif].sample(n=min(100_000, len(df)), random_state=config.RANDOM_STATE)
vif = calcular_vif(muestra)
vif["alerta_vif"] = vif["vif"] > config.UMBRAL_VIF

validacion = validacion.merge(vif, on="variable", how="left")

print(vif.sort_values("vif", ascending=False).head(15).to_string(index=False))
print(f"\nVariables con VIF > {config.UMBRAL_VIF}: "
      f"{vif.loc[vif['alerta_vif'], 'variable'].tolist()}")

triada = vif[vif["variable"].isin(["total_patrimonio", "total_activos", "total_pasivos"])]
print("\nVIF de la triada contable patrimonio/activos/pasivos (patrimonio = "
      "activos - pasivos, SPEC_V2 SS4.5):")
print(triada.to_string(index=False))
if not triada.empty and (triada["vif"] > config.UMBRAL_VIF).any():
    print("-> Como se esperaba: la relacion contable dispara el VIF entre ellas. "
          "Eso NO es un bug, es la identidad activos-pasivos=patrimonio operando.")
else:
    print("-> En ESTA muestra la triada sale con VIF bajo (no dispara la alerta); "
          "lo que sí se dispara es otro cluster: ratios y derivadas financieras "
          "(p.ej. capacidad_ahorro, pct_ahorro_ingreso, ratio_egreso_ingreso, "
          "dif_ingreso_declarado_estimado) que se calculan por formula DIRECTA a "
          "partir de ingresos_mensuales/total_egresos_mensuales, tambien presentes "
          "en la muestra -- es la misma clase de relacion estructural esperada "
          "(variables ligadas por definicion, no por azar), solo que no es la "
          "triada puntual que el spec menciona como ejemplo. Tampoco es un bug: "
          "es exactamente el tipo de alerta que SS4.5 pide reportar, no ocultar.")


                       variable        vif  alerta_vif
       bolsillos_saldo_snapshot        inf        True
               capacidad_ahorro        inf        True
             cdt_saldo_snapshot        inf        True
            saldo_liquido_total        inf        True
             ingresos_mensuales        inf        True
             pct_ahorro_ingreso        inf        True
           ratio_egreso_ingreso        inf        True
      fiducuenta_saldo_snapshot        inf        True
              estimador_ingreso        inf        True
 dif_ingreso_declarado_estimado        inf        True
cuenta_corriente_saldo_snapshot        inf        True
   cuenta_ahorro_saldo_snapshot        inf        True
        total_egresos_mensuales        inf        True
    saldo_invertido_no_etiqueta        inf        True
              cdt_saldo_prom_6m 514.114158        True

Variables con VIF > 10.0: ['bolsillos_saldo_prom_6m', 'bolsillos_saldo_snapshot', 'capacidad_ahorro', 'cdt_saldo_prom_6

In [5]:
from src.decisiones import decidir_tratamiento_vivienda

# IV de la categorica CON "Sin dato" como un bin mas (ya viene asi de la celda 1)
iv_vivienda = float(validacion.loc[validacion["variable"] == "desc_tipo_de_vivienda", "iv"].iloc[0])
# IV de la bandera binaria por separado (SPEC_V2 SS4, ultimo parrafo)
iv_bandera, _ = calcular_woe_iv(df["tiene_dato_vivienda"], y)

decision_viv = decidir_tratamiento_vivienda(iv_vivienda, iv_bandera)

# Verificacion previa obligatoria de SS6.5: tiene dato de vivienda codifica
# vinculacion crediticia en vez de patrimonio?
comparacion = []
for col in ["total_patrimonio", "ingresos_mensuales", "n_productos_no_etiqueta"]:
    mw = mann_whitney(df[col], df["tiene_dato_vivienda"])
    comparacion.append({"variable": col,
                        "mediana_con_dato": mw["mediana_evento"],
                        "mediana_sin_dato": mw["mediana_no_evento"],
                        "p_valor": mw["p_valor"]})
comparacion = pd.DataFrame(comparacion)

tasa_por_nivel = (
    df.groupby("desc_tipo_de_vivienda")["etiqueta_adopcion"]
    .agg(n_clientes="count", tasa_adopcion="mean").reset_index()
)

print(f"IV categorica (con 'Sin dato'): {iv_vivienda:.4f}")
print(f"IV bandera tiene_dato_vivienda: {iv_bandera:.4f}")
print(f"DECISION: {decision_viv['accion']}\n")
print("Sesgo de captura -- comparacion con/sin dato (SS6.5, verificacion previa):")
print(comparacion.to_string(index=False))
print("\nTasa de adopcion por nivel (incluido 'Sin dato'):")
print(tasa_por_nivel.to_string(index=False))

with open(config.OUTPUTS_DIR / "eda" / "decision_vivienda.json", "w", encoding="utf-8") as f:
    json.dump({**decision_viv,
               "comparacion_sesgo_captura": comparacion.to_dict(orient="records"),
               "tasa_por_nivel": tasa_por_nivel.to_dict(orient="records")},
              f, indent=2, ensure_ascii=False, default=str)

registrar_decision(
    clave="inclusion_vivienda",
    decision=decision_viv["accion"],
    motivo=f"IV categorica={iv_vivienda:.4f}, IV bandera={iv_bandera:.4f}, "
           f"umbral={config.UMBRAL_IV_MINIMO} (SPEC_V2 SS6.5)",
    evidencia=decision_viv,
)

IV categorica (con 'Sin dato'): 0.1207
IV bandera tiene_dato_vivienda: 0.1083
DECISION: conservar_categorica_con_sin_dato

Sesgo de captura -- comparacion con/sin dato (SS6.5, verificacion previa):
               variable  mediana_con_dato  mediana_sin_dato  p_valor
       total_patrimonio        24000000.0         3952000.0      0.0
     ingresos_mensuales         2500000.0         1938493.5      0.0
n_productos_no_etiqueta               1.0               1.0      0.0

Tasa de adopcion por nivel (incluido 'Sin dato'):
desc_tipo_de_vivienda  n_clientes  tasa_adopcion
            ARRENDADA       45949       0.072624
             FAMILIAR      129359       0.111256
           NO INFORMA       18190       0.091974
               PROPIA       75034       0.118959
             Sin dato      591691       0.056293


WindowsPath('C:/Users/natam/OneDrive/Desktop/Prueba-Tecnica-CREAN/.claude/worktrees/pipeline-crean-sdd/outputs/decisiones/log_decisiones.csv')

In [6]:
def decidir_inclusion(fila):
    # SPEC_V2 SS6.4: genero se mide pero NUNCA entra, por criterio de idoneidad
    # financiera (acceso historico desigual), no porque su poder predictivo sea
    # bajo -- si resultara predictivo, seria por esa desigualdad, no por
    # propension genuina.
    if fila["variable"] == "desc_genero":
        return "excluida_por_idoneidad_no_por_poder_predictivo"
    if fila["variable"] == "desc_tipo_de_vivienda":
        return ("incluir" if decision_viv["conservar_categorica"]
                else "excluir_categorica_" + ("usar_bandera" if decision_viv["conservar_bandera"]
                                              else "descartar_bloque"))
    if fila["variable"] in ("tiene_dato_vivienda", "falta_vivienda"):
        # `falta_vivienda` = 1 - `tiene_dato_vivienda` (complemento exacto, ver
        # oro/construir_cliente_features.py) y con "Sin dato" como bin propio
        # de la categorica, `tiene_dato_vivienda` es ademas derivable de
        # `desc_tipo_de_vivienda` (!= "Sin dato"): las tres cosas cargan la
        # MISMA informacion, por eso sus IV salen casi identicos (0.1207 vs
        # 0.1083). SS6.5 describe resultados mutuamente excluyentes -- "or
        # conservar_categorica" aqui mantendria la bandera viva en el caso mas
        # comun (conservar la categorica) pese a que la decision dice
        # explicitamente que NO se necesita la bandera por separado. Se
        # conservan (cualquiera de las dos) unicamente si decidir_tratamiento_vivienda
        # dijo que la bandera por si sola es la que aporta.
        return "incluir" if decision_viv["conservar_bandera"] else "descartar"
    if fila["clase_iv"] == "descartar":
        return "descartar_iv_insuficiente"
    # OJO: para las categoricas `alerta_vif` es NaN (el VIF solo se calcula sobre
    # continuas), y `bool(nan)` es True. Hay que comparar explicitamente.
    if fila.get("alerta_vif") is True or fila.get("alerta_vif") == True:  # noqa: E712
        return "incluir_con_alerta_multicolinealidad"
    return "incluir"

validacion["decision_inclusion"] = validacion.apply(decidir_inclusion, axis=1)

orden = ["variable", "tipo", "n_nulos", "iv", "clase_iv", "p_mann_whitney",
         "chi2", "p_chi2", "v_cramer", "p_valor", "q_bh", "significativa_fdr",
         "vif", "alerta_vif", "decision_inclusion"]
validacion = validacion[[c for c in orden if c in validacion.columns]]
validacion = validacion.sort_values("iv", ascending=False)
validacion.to_csv(config.OUTPUTS_DIR / "eda" / "validacion_variables.csv", index=False)

# WoE por bin solo de las variables que se conservan (SPEC_V2 SS4.1)
conservadas = set(validacion.loc[
    validacion["decision_inclusion"].str.startswith("incluir"), "variable"])
woe_por_bin[woe_por_bin["variable"].isin(conservadas)].to_csv(
    config.OUTPUTS_DIR / "eda" / "woe_por_bin.csv", index=False)

descartadas = validacion.loc[
    validacion["decision_inclusion"] == "descartar_iv_insuficiente", "variable"].tolist()
registrar_decision(
    clave="variables_descartadas_por_iv",
    decision=f"{len(descartadas)} variables descartadas",
    motivo=f"IV < {config.UMBRAL_IV_MINIMO} (SPEC_V2 SS4.1)",
    evidencia={"variables": descartadas},
)

print(validacion.to_string(index=False))
print(f"\nConservadas: {len(conservadas)} | Descartadas por IV: {len(descartadas)}")
print("\nSPEC_V2 SS6.4 -- desc_genero se midio y se reporta, pero su exclusion esta "
      "decidida por criterio de idoneidad financiera, NO por su poder predictivo.")
print(f"\nSS6.5 -- tiene_dato_vivienda: "
      f"{validacion.loc[validacion['variable']=='tiene_dato_vivienda','decision_inclusion'].iloc[0]} | "
      f"falta_vivienda: "
      f"{validacion.loc[validacion['variable']=='falta_vivienda','decision_inclusion'].iloc[0]} "
      f"(derivado de decision_viv['conservar_bandera']={decision_viv['conservar_bandera']}, "
      "sin el 'or conservar_categorica' que las mantenia vivas por error).")


                              variable       tipo  n_nulos       iv  clase_iv  p_mann_whitney         chi2  p_chi2  v_cramer       p_valor          q_bh significativa_fdr        vif alerta_vif                             decision_inclusion
               n_productos_no_etiqueta   continua        0 2.149881    fuerte    0.000000e+00          NaN     NaN       NaN  0.000000e+00  0.000000e+00              True   1.130291      False                                        incluir
                   saldo_liquido_total   continua        0 1.838402    fuerte    0.000000e+00          NaN     NaN       NaN  0.000000e+00  0.000000e+00              True        inf       True           incluir_con_alerta_multicolinealidad
             antiguedad_relacion_meses   continua   331037 1.615513    fuerte    0.000000e+00          NaN     NaN       NaN  0.000000e+00  0.000000e+00              True   1.035976      False                                        incluir
                dias_desde_ultimo_dato  